# ArcGIS Online Item Terms of Use Editor

**Welcome!**  
This notebook is designed for ArcGIS Online administrators who need to review and update item Terms of Use at scale. It focuses on the Terms of Use field (`licenseInfo`) and supports a review-first workflow before any edits are applied.
This version bundles `helper_functions.py` and `Esri_ToU.html` template directly into the notebook. During setup, those assets are expanded into `/arcgis/home/notebook_outputs` (or local notebook output paths), where you can inspect inputs and outputs as you work. A review webpage is generated so operators can verify planned changes before execution.
*** BE CAUTIOUS WITH ANY TOOL LIKE THIS THAT BULK EDITS ITEMS *** The workflow intentionally requires explicit review and confirmation before updates.

**Where this notebook can run**  
- ArcGIS Online Notebook (JupyterLab-style).
- VS Code on macOS with a local Jupyter kernel.
- VS Code on Windows with a local Jupyter kernel.

**How to use this notebook**  
 - Click on the text "Setup and authenticate" below. 
 - There are two types of cells, Markdown (formatted notes) and Code.
 - An indicator -- typically a vertical blue line -- should highlight that you have selected the "Setup and authenticate" Markdown cell.
 - Once selected, click the "Play" button in the toolbar above to run the cell and advance to the next Code cell.
 - Click the "Play" button a second time to run the code cell.
 - After several seconds a "Setup Notebook" button should appear. Click the button to begin setup and authentication.
 - After each cell completes, click the text within the following Markdown cell.
 - Click the "Play" button to advance to the Code cell, then click the "Play" button a second time to make a button appear.
 - Click the button to run the code in the cell. 

**Notes**  
- Organization-wide scans can take time, especially in large orgs, so progress messages are shown as users are processed.
- Candidate search and structural replacement are separate steps by design:
  - Scan steps find candidate items that contain the terms you specify.
  - Dry-run applies structural matching logic to build the replacement plan.
- You can monitor the status of long running cells by viewing the small circle in the top right of the page.
- If you click on a code cell it will expand showing you the behind-the-scenes Python code.
- For a cleaner interface select View > Collapse All Code in the menu bar above to hide the code .
- If at any point you get stuck and want to start over, just click Kernel > Restart Kernel and Clear Outputs of All Cells... in the menu bar
- The workflow is designed to be safe by default: review first, then update.

**TL;DR**

In [ ]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does (admin workflow)**  
- Authenticates to ArcGIS Online.
- Scans an entire organization's item Terms of Use (`licenseInfo`) for operator-defined terms.
- Identifies candidate items first, then separately builds a structural dry-run replacement plan.
- Exports scan outputs for auditability (default filenames: `scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans for additional terms while excluding already matched item IDs.
- Generates a side-by-side HTML review report for operator QA.
- Applies updates only after explicit confirmation.
- Exports success and error results for record-keeping.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))

## 1. Setup and authenticate
Connect to ArcGIS Online and initialize the notebook environment.

In [ ]:
# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

NOTEBOOK_DIR = Path.cwd().resolve()
IS_PORTABLE_NOTEBOOK = "RUNTIME_DIR" in globals()

if IS_PORTABLE_NOTEBOOK:
    # Portable notebook: prefer freshly bootstrapped assets in notebook_outputs.
    candidate_helper_dirs = [
        Path(globals()["RUNTIME_DIR"]).resolve(),
        NOTEBOOK_DIR / "notebook_outputs",
        NOTEBOOK_DIR,
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]
else:
    # Source notebook: prefer repository files first to avoid stale bootstrapped copies.
    candidate_helper_dirs = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR / ".local_testing" / "notebook_outputs",
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]

selected_helper_dir = None
for p in candidate_helper_dirs:
    helper_file = p / "helper_functions.py"
    if helper_file.exists():
        selected_helper_dir = p.resolve()
        break

if selected_helper_dir is None:
    raise FileNotFoundError(
        "Could not locate helper_functions.py in expected locations. "
        "For source notebook runs, keep helper_functions.py in the notebook folder. "
        "For portable runs, execute the bootstrap section first."
    )

# Ensure the selected helper path wins over stale entries.
selected_helper_dir_str = str(selected_helper_dir)
sys.path[:] = [x for x in sys.path if x != selected_helper_dir_str]
sys.path.insert(0, selected_helper_dir_str)

# Force reload source if helper_functions was previously imported from another location.
if "helper_functions" in sys.modules:
    del sys.modules["helper_functions"]

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    bind_button_with_status,
    bind_primary_scan_with_cancel,
    setup_notebook_btn,
    save_scan_outputs_btn,
    save_secondary_scan_outputs_btn,
    load_previous_scan_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    run_dry_run_with_file_btn,
    preview_dry_run_match_btn,
    create_report_btn,
    export_dry_run_btn,
    load_update_selection_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
    )

# Resolve the canonical ToU template path based on notebook mode.
if IS_PORTABLE_NOTEBOOK:
    resolved_tou_path = (selected_helper_dir / "Esri_ToU.html").resolve()
else:
    resolved_tou_path = (NOTEBOOK_DIR / "Esri_ToU.html").resolve()
if not resolved_tou_path.exists():
    resolved_tou_path = Path(OFFICIAL_TOU_HTML_FILE).resolve()

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": str(resolved_tou_path),
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

if not IS_PORTABLE_NOTEBOOK:
    print(f"Helper path: {selected_helper_dir}")
    print(f"Official ToU template file: {context['official_tou_html_file']}")

setup_output = initialize_ui("output")
context["setup_output"] = setup_output
auth_container = widgets.VBox([])
context["auth_container"] = auth_container

# Create widgets
setup_button = initialize_ui("button", description="Setup Notebook", width="200px")
setup_status = widgets.HTML(value="")
context["setup_status"] = setup_status
display(widgets.HBox([setup_button, setup_status]))
bind_button_with_status(
    setup_button,
    setup_notebook_btn,
    "setup_status",
    "Setup in progress... please wait.",
    "Setup complete.",
    "Setup failed. See details below.",
    output_key="setup_output",
)
display(setup_output)
display(auth_container)

## 2. Scan your content
Search your organization for candidate items containing the text strings and/or HTML entered below.
This step is candidate discovery only; structural replacement matching happens later in the dry-run step.
There is an optional cap: leave it blank to scan the entire org, or enter a number to stop after that many matches for faster test runs.
After the scan finishes, optional save fields appear below when there is scan output worth exporting.

In [ ]:
# Cell 2: Scan org content for licenseInfo matches and optionally save the results
scan_output = initialize_ui("output")
context["scan_output"] = scan_output

help2 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt; link &lt;/a&gt;&quot;.<br>"
        "Formatting will automatically be bundled for processing when you click the button."
        "</div>"
    )
)

scan_terms_input = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["scan_terms_input"] = scan_terms_input
scan_limit_input = initialize_ui(
    "text",
    value="",
    description="Stop scan after X matches (optional):",
    width="300px",
)
context["scan_limit_input"] = scan_limit_input
scan_button = initialize_ui("button", description="Scan for items", width="200px")
scan_status = widgets.HTML(value="")
context["scan_status"] = scan_status

save_scan_output = initialize_ui("output")
context["save_scan_output"] = save_scan_output
scan_matches_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Save matching items to CSV:",
    width="700px",
)
context["scan_matches_path_input"] = scan_matches_path_input
scan_errors_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Save errors to CSV:",
    width="700px",
)
context["scan_errors_path_input"] = scan_errors_path_input
scan_all_items_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="Save scanned items to CSV:",
    width="700px",
)
context["scan_all_items_path_input"] = scan_all_items_path_input
save_scan_button = initialize_ui("button", description="Save files")
save_scan_status = widgets.HTML(value="")
context["save_scan_status"] = save_scan_status
save_scan_container = widgets.VBox([])
context["save_scan_container"] = save_scan_container

def refresh_scan_save_ui():
    matches_df = context.get("matches_df")
    errors_df = context.get("errors_df")
    all_items_df = context.get("all_items_df")

    if matches_df is None and errors_df is None and all_items_df is None:
        save_scan_container.children = ()
        return

    step3_children = [widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Save scan outputs to save time in a future run.</div>")]
    if matches_df is not None and not matches_df.empty:
        step3_children.append(scan_matches_path_input)
    if errors_df is not None and not errors_df.empty:
        step3_children.append(scan_errors_path_input)
    if all_items_df is not None and not all_items_df.empty:
        step3_children.append(scan_all_items_path_input)

    if len(step3_children) == 1:
        step3_children.append(
            widgets.HTML(
                value="<div style='margin:0; padding:0;'>No non-empty scan output files are available to save.</div>"
            )
        )
    else:
        step3_children.append(widgets.HBox([save_scan_button, save_scan_status]))

    step3_children.append(save_scan_output)
    save_scan_container.children = tuple(step3_children)

context["refresh_scan_save_ui"] = refresh_scan_save_ui
refresh_scan_save_ui()

display(
    widgets.VBox([
        help2,
        scan_terms_input,
        scan_limit_input,
        widgets.HBox([scan_button, scan_status]),
        scan_output,
        save_scan_container,
    ])
)

bind_primary_scan_with_cancel(
    scan_button,
    status_key="scan_status",
    output_key="scan_output",
    input_key="scan_terms_input",
    limit_input_key="scan_limit_input",
)

bind_button_with_status(
    save_scan_button,
    save_scan_outputs_btn,
    "save_scan_status",
    "Save in progress... please wait.",
    "Save complete.",
    "Save failed. See details below.",
    output_key="save_scan_output",
)

## 3. Optionally reload results from a previous scan
Load previously saved CSV files so you can continue working without running a new scan.

In [ ]:
# Cell 3: Optionally load results from a previous scan
reload_scan_output = initialize_ui("output")
context["reload_scan_output"] = reload_scan_output

reload_matches_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="900px",
)
context["reload_matches_path_input"] = reload_matches_path_input
reload_errors_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="900px",
)
context["reload_errors_path_input"] = reload_errors_path_input
reload_all_items_path_input = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="900px",
)
context["reload_all_items_path_input"] = reload_all_items_path_input
reload_scan_button = initialize_ui("button", description="Load previous scan files", width="240px")
reload_scan_status = widgets.HTML(value="")
context["reload_scan_status"] = reload_scan_status

display(
    widgets.VBox([
        reload_matches_path_input,
        reload_errors_path_input,
        reload_all_items_path_input,
        widgets.HBox([reload_scan_button, reload_scan_status]),
        reload_scan_output,
    ])
)

bind_button_with_status(
    reload_scan_button,
    load_previous_scan_btn,
    "reload_scan_status",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="reload_scan_output",
)

## 4. Secondary scan
This step saves you time on the primary run if you want to search additional terms.
If you want to search again, click the check box and add the new terms to the input box.
After the secondary scan finishes, optional save fields appear below when there is secondary scan output worth exporting.

In [ ]:
# Cell 4: Optional secondary scan using new terms and optionally save the results
secondary_scan_output = initialize_ui("output")
context["secondary_scan_output"] = secondary_scan_output
secondary_scan_enabled_checkbox = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["secondary_scan_enabled_checkbox"] = secondary_scan_enabled_checkbox
help5 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "As above, use CSV-style input separated by commas.<br>"
        "Wrap text with spaces in double quotes, for example: &quot;text with spaces&quot;."
        "</div>"
    )
)
secondary_scan_terms_input = initialize_ui(
    "textarea",
    value='https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg',
    description="",
    width="700px",
    height="70px",
)
context["secondary_scan_terms_input"] = secondary_scan_terms_input

secondary_scan_button = initialize_ui("button", description="Run secondary scan")
secondary_scan_status = widgets.HTML(value="")
context["secondary_scan_status"] = secondary_scan_status

save_secondary_scan_output = initialize_ui("output")
context["save_secondary_scan_output"] = save_secondary_scan_output
secondary_matches_path_input = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_matches.csv"),
    description="Secondary matches CSV:",
    width="700px",
)
context["secondary_matches_path_input"] = secondary_matches_path_input
secondary_errors_path_input = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_errors.csv"),
    description="Secondary errors CSV:",
    width="700px",
)
context["secondary_errors_path_input"] = secondary_errors_path_input
secondary_all_items_path_input = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_all_items.csv"),
    description="Secondary all items CSV:",
    width="700px",
)
context["secondary_all_items_path_input"] = secondary_all_items_path_input
save_secondary_scan_button = initialize_ui("button", description="Save secondary files")
save_secondary_scan_status = widgets.HTML(value="")
context["save_secondary_scan_status"] = save_secondary_scan_status
save_secondary_scan_container = widgets.VBox([])
context["save_secondary_scan_container"] = save_secondary_scan_container

def refresh_secondary_save_ui():
    new_matches_df = context.get("new_matches_df")
    new_errors_df = context.get("new_errors_df")
    new_all_items_df = context.get("new_all_items_df")

    if new_matches_df is None and new_errors_df is None and new_all_items_df is None:
        save_secondary_scan_container.children = ()
        return

    step6_children = [widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Save current secondary scan outputs</div>")]
    if new_matches_df is not None and not new_matches_df.empty:
        step6_children.append(secondary_matches_path_input)
    if new_errors_df is not None and not new_errors_df.empty:
        step6_children.append(secondary_errors_path_input)
    if new_all_items_df is not None and not new_all_items_df.empty:
        step6_children.append(secondary_all_items_path_input)

    if len(step6_children) == 1:
        step6_children.append(
            widgets.HTML(
                value="<div style='margin:0; padding:0;'>No non-empty secondary scan output files are available to save.</div>"
            )
        )
    else:
        step6_children.append(widgets.HBox([save_secondary_scan_button, save_secondary_scan_status]))

    step6_children.append(save_secondary_scan_output)
    save_secondary_scan_container.children = tuple(step6_children)

context["refresh_secondary_save_ui"] = refresh_secondary_save_ui
refresh_secondary_save_ui()

display(
    widgets.VBox([
        secondary_scan_enabled_checkbox,
        help5,
        secondary_scan_terms_input,
        widgets.HBox([secondary_scan_button, secondary_scan_status]),
        secondary_scan_output,
        save_secondary_scan_container,
    ])
)

bind_button_with_status(
    secondary_scan_button,
    run_secondary_scan_btn,
    "secondary_scan_status",
    "Secondary scan in progress... please wait.",
    "Secondary scan complete.",
    "Secondary scan failed. See details below.",
    output_key="secondary_scan_output",
)

bind_button_with_status(
    save_secondary_scan_button,
    save_secondary_scan_outputs_btn,
    "save_secondary_scan_status",
    "Secondary save in progress... please wait.",
    "Secondary save complete.",
    "Secondary save failed. See details below.",
    output_key="save_secondary_scan_output",
)

## 5. Optionally narrow your original query
You can filter the scan results to show only items containing the exact text entered below.

In [ ]:
# Cell 5: Optionally filter the scan result to rows containing the exact text below
exact_match_output = initialize_ui("output")
context["exact_match_output"] = exact_match_output
exact_match_input = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact text:",
    width="700px",
)
context["exact_match_input"] = exact_match_input
exact_match_button = initialize_ui("button", description="Filter exact matches", width="200px")
exact_match_status = widgets.HTML(value="")
context["exact_match_status"] = exact_match_status
display(widgets.VBox([exact_match_input, widgets.HBox([exact_match_button, exact_match_status]), exact_match_output]))

bind_button_with_status(
    exact_match_button,
    run_strict_match_filter_btn,
    "exact_match_status",
    "Filter in progress... please wait.",
    "Filter complete.",
    "Filter failed. See details below.",
    output_key="exact_match_output",
)

## 6. Dry run
Do a dry-run before making any changes. Modify the input file to use your own custom HTML that will replace the search terms.
By default, the edit logic uses a semi-greedy matcher: it looks for a recognized Esri logo, then scans forward within bounded distance for the license text and the related links. After a replacement is made, cleanup includes removing trivial wrapper junk around the replacement block.

If you enable strict matching below, the dry run will only match blocks that contain your search text exactly. Use strict mode when you want a more conservative replacement plan.

After the dry run finishes, an optional CSV export appears when there is output worth saving.

In [ ]:
# Cell 6: Do a dry-run before making any changes and optionally export the plan
dry_run_output = initialize_ui("output")
context["dry_run_output"] = dry_run_output
dry_run_preview_output = initialize_ui("output")
context["dry_run_preview_output"] = dry_run_preview_output
dry_run_template_path_input = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
context["dry_run_template_path_input"] = dry_run_template_path_input
checkbox8 = initialize_ui(
    "checkbox",
    description="Enforce strict matching?",
    value=False,
    width="320px",
)
context["checkbox8"] = checkbox8
dry_run_preview_button = initialize_ui("button", description="Preview card comparison", width="220px")
dry_run_preview_status = widgets.HTML(value="")
context["dry_run_preview_status"] = dry_run_preview_status
dry_run_button = initialize_ui("button", description="Do a dry run", width="180px")
dry_run_status = widgets.HTML(value="")
context["dry_run_status"] = dry_run_status

dry_run_export_output = initialize_ui("output")
context["dry_run_export_output"] = dry_run_export_output
dry_run_export_path_input = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["dry_run_export_path_input"] = dry_run_export_path_input
dry_run_export_button = initialize_ui("button", description="Export dry-run CSV", width="200px")
dry_run_export_status = widgets.HTML(value="")
context["dry_run_export_status"] = dry_run_export_status
export9_container = widgets.VBox([])
context["export9_container"] = export9_container

def refresh_dry_run_export_ui():
    plan_df = context.get("plan_df")

    if plan_df is None:
        export9_container.children = ()
        return

    export9_container.children = (
        widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Export current dry-run results</div>"),
        dry_run_export_path_input,
        widgets.HBox([dry_run_export_button, dry_run_export_status]),
        dry_run_export_output,
    )

context["refresh_dry_run_export_ui"] = refresh_dry_run_export_ui
refresh_dry_run_export_ui()

display(
    widgets.VBox([
        dry_run_template_path_input,
        checkbox8,
        widgets.HBox([dry_run_preview_button, dry_run_preview_status]),
        dry_run_preview_output,
        widgets.HBox([dry_run_button, dry_run_status]),
        dry_run_output,
        export9_container,
    ])
)

bind_button_with_status(
    dry_run_preview_button,
    preview_dry_run_match_btn,
    "dry_run_preview_status",
    "Preview generation in progress... please wait.",
    "Preview ready.",
    "Preview failed. See details below.",
    output_key="dry_run_preview_output",
)

bind_button_with_status(
    dry_run_button,
    run_dry_run_with_file_btn,
    "dry_run_status",
    "Dry run in progress... please wait.",
    "Dry run complete.",
    "Dry run failed. See details below.",
    output_key="dry_run_output",
)

bind_button_with_status(
    dry_run_export_button,
    export_dry_run_btn,
    "dry_run_export_status",
    "Dry-run export in progress... please wait.",
    "Dry-run export complete.",
    "Dry-run export failed. See details below.",
    output_key="dry_run_export_output",
)

## 7. Create a report to review before committing the edits
A HTML file will be created. Large reports cannot be properly displayed in the output cell. When this happens, download the file from /arcgis/home/notebook_outputs by clicking on the filename. Then open that file in your local browser. You'll then make selections, save a JSON file and upload that file to /arcgis/home/notebook_outputs for the final editing step.
There is an optional cap: leave it blank to include all planned updates, or enter a number to generate a smaller review report for faster testing.

In [ ]:
# Cell 7: Generate an HTML report for review before updating items
create_report_output = initialize_ui("output")
context["create_report_output"] = create_report_output
report_path_input = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["report_path_input"] = report_path_input
selection_json_name_input = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Filename generated when downloading IDs after review:",
    width="700px",
)
context["selection_json_name_input"] = selection_json_name_input
create_report_button = initialize_ui("button", description="Create report", width="200px")
create_report_status = widgets.HTML(value="")
context["create_report_status"] = create_report_status

display(
    widgets.VBox([
        report_path_input,
        selection_json_name_input,
        widgets.HBox([create_report_button, create_report_status]),
        create_report_output,
    ])
)

bind_button_with_status(
    create_report_button,
    create_report_btn,
    "create_report_status",
    "Report generation in progress... please wait.",
    "Report generation complete.",
    "Report generation failed. See details below.",
    output_key="create_report_output",
)

## 8. Commit updates
Use this step to safely confirm what will be edited before making any changes.
1. Enter the JSON or CSV file path that contains item IDs selected from the report. (the default file will be preloaded)
2. Click **Load file** to preview how many items will be edited.
3. Review the summary shown in the output area.
4. If it looks correct, type `APPLY UPDATES` in the confirmation box.
5. Click **Execute update** to apply the edits.

In [ ]:
# Cell 8: Apply updates only after reviewing the dry run report 
apply_edits_output = initialize_ui("output")
context["apply_edits_output"] = apply_edits_output
selected_ids_to_edit_path_input = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="JSON file with item IDs to update:",
    width="700px",
)
context["selected_ids_to_edit_path_input"] = selected_ids_to_edit_path_input

load_selected_ids_button = initialize_ui("button", description="Load file", width="140px")
load_selected_ids_status = widgets.HTML(value="")
context["load_selected_ids_status"] = load_selected_ids_status

apply_edits_confirmation_input = initialize_ui(
    "text",
    value="",
    description="Type APPLY UPDATES to confirm:",
    width="700px",
)
context["apply_edits_confirmation_input"] = apply_edits_confirmation_input

apply_edits_button = initialize_ui("button", description="Execute update", width="180px")
apply_edits_status = widgets.HTML(value="")
context["apply_edits_status"] = apply_edits_status
display(
    widgets.VBox([
        selected_ids_to_edit_path_input,
        widgets.HBox([load_selected_ids_button, load_selected_ids_status]),
        apply_edits_output,
        apply_edits_confirmation_input,
        widgets.HBox([apply_edits_button, apply_edits_status]),
    ])
)

bind_button_with_status(
    load_selected_ids_button,
    load_update_selection_btn,
    "load_selected_ids_status",
    "Loading file and previewing selection... please wait.",
    "File loaded. Review summary below.",
    "Load failed. See details below.",
    output_key="apply_edits_output",
)

bind_button_with_status(
    apply_edits_button,
    apply_updates_btn,
    "apply_edits_status",
    "Update in progress... please wait.",
    "Update complete.",
    "Update failed. See details below.",
    output_key="apply_edits_output",
)

## 9. Export results of the editing process
Export csv files for record-keeping.

In [ ]:
# Cell 9: Export final update results to CSV files for record-keeping
export_final_results_output = initialize_ui("output")
context["export_final_results_output"] = export_final_results_output
edit_successes_path_input = initialize_ui(
    "text",
    value=default_output_path_str("update_successes.csv"),
    description="Success CSV:",
    width="700px",
)
context["edit_successes_path_input"] = edit_successes_path_input
edit_errors_path_input = initialize_ui(
    "text",
    value=default_output_path_str("update_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["edit_errors_path_input"] = edit_errors_path_input
export_final_results_button = initialize_ui("button", description="Export final CSVs", width="180px")
export_final_results_status = widgets.HTML(value="")
context["export_final_results_status"] = export_final_results_status

success_df = context.get("success_df")
update_errors_df = context.get("update_errors_df")

step12_children = []
if success_df is not None and not success_df.empty:
    step12_children.append(edit_successes_path_input)
if update_errors_df is not None and not update_errors_df.empty:
    step12_children.append(edit_errors_path_input)

if not step12_children:
    step12_children.append(
        widgets.HTML(
            value="<div style='margin:0; padding:0;'>No non-empty final result files are available to export yet.</div>"
        )
    )
else:
    step12_children.append(widgets.HBox([export_final_results_button, export_final_results_status]))

step12_children.append(export_final_results_output)
display(widgets.VBox(step12_children))

bind_button_with_status(
    export_final_results_button,
    export_final_results_btn,
    "export_final_results_status",
    "Final export in progress... please wait.",
    "Final export complete.",
    "Final export failed. See details below.",
    output_key="export_final_results_output",
)